In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pyarrow

# caminhos relativos à raiz do projeto
RATINGS_PATH = "data/raw/ml-latest-small/ratings.csv"
MOVIES_PATH  = "data/raw/ml-latest-small/movies.csv"

ratings = pd.read_csv(RATINGS_PATH)
movies  = pd.read_csv(MOVIES_PATH)

print("ratings:", ratings.shape)
print("movies: ", movies.shape)
movies.head()

ratings: (100836, 4)
movies:  (9742, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [2]:
# Partindo do título do filme, pegamos o ano de lançamento (que vem entre parênteses ao final do titulo)
def get_ano(titulo):
    match = re.search(r"\((\d{4})\)$", titulo.strip())
    return int(match.group(1)) if match else None

# Removemos o ano do título
def clean_titulo(titulo):
    return re.sub(r"\s*\(\d{4}\)\s*$", "", titulo.strip())

# Trocamos o separador de gêneros de "|" por ","
def clean_generos(generos):
    return generos.replace("|", ", ")

movies["year"] = movies["title"].apply(get_ano)
movies["title_clean"] = movies["title"].apply(clean_titulo)
movies["genres"] = movies["genres"].apply(clean_generos)

print(movies[["title", "title_clean", "year", "genres"]].tail(10))

                                                  title  \
9732                          Gintama: The Movie (2010)   
9733  anohana: The Flower We Saw That Day - The Movi...   
9734                                Silver Spoon (2014)   
9735            Love Live! The School Idol Movie (2015)   
9736           Jon Stewart Has Left the Building (2015)   
9737          Black Butler: Book of the Atlantic (2017)   
9738                       No Game No Life: Zero (2017)   
9739                                       Flint (2017)   
9740                Bungo Stray Dogs: Dead Apple (2018)   
9741                Andrew Dice Clay: Dice Rules (1991)   

                                          title_clean    year  \
9732                               Gintama: The Movie  2010.0   
9733  anohana: The Flower We Saw That Day - The Movie  2013.0   
9734                                     Silver Spoon  2014.0   
9735                 Love Live! The School Idol Movie  2015.0   
9736                Jon S

In [3]:
# DECISÃO DE LIMPEZA - filmes sem ano
# Alguns filmes foram removidos por não possuírem ano no título.
# Alternativas consideradas e descartadas:
#   - imputar ano da primeira avaliação: incorreto, filmes antigos podem ter sido avaliados muito depois do lançamento.
#   - buscar manualmente: volume pequeno, custo não justifica.
sem_ano_ids = movies[movies["year"].isnull()]["movieId"]
movies_clean = movies[movies["year"].notna()].copy()

print(f"Filmes removidos por ausência de ano: {len(sem_ano_ids)}")
print(f"Filmes restantes: {len(movies_clean)}")
print(f"Porcentagem do dataset descartado: {len(sem_ano_ids)/len(movies_clean)*100}")

Filmes removidos por ausência de ano: 13
Filmes restantes: 9729
Porcentagem do dataset descartado: 0.13362113269606332


In [4]:
# DECISÃO DE LIMPEZA - filmes sem gênero
# alguns filmes removidos por ausência de gênero ("(no genres listed)").
# Alternativas consideradas e descartadas:
#   - imputar gênero pelo título: incorreto. Ex: "Nightmare Before Christmas" sugere terror mas é animação/musical.
#   - buscar em API externa (TMDB): possível, mas fora do escopo do MVP.

movies_clean = movies_clean[movies_clean["genres"] != "(no genres listed)"].copy()
sem_genero = movies[movies["genres"] == "(no genres listed)"]

print(f"Filmes removidos por ausência de gênero: {(movies['genres'] == '(no genres listed)').sum()}")
print(f"Filmes restantes: {len(movies_clean)}")
print(f"Porcentagem do dataset descartado: {len(sem_ano_ids)/len(movies_clean)*100}")

Filmes removidos por ausência de gênero: 34
Filmes restantes: 9704
Porcentagem do dataset descartado: 0.1339653751030503


In [5]:
# DECISÃO DE LIMPEZA - reviews dos filmes removidos
# Como alguns filmes foram removidos, não faz sentido manter as reviews desses filmes.

filmes_validos = set(movies_clean["movieId"])
ratings_clean = ratings[ratings["movieId"].isin(filmes_validos)].copy()

print(f"Avaliações removidas por ausência de filme: {len(ratings) - len(ratings_clean):,}")
print(f"Avaliações restantes: {len(ratings_clean):,}")
print(f"Porcentagem descartada: {(len(ratings) - len(ratings_clean))/len(ratings_clean)*100}")

Avaliações removidas por ausência de filme: 55
Avaliações restantes: 100,781
Porcentagem descartada: 0.05457377878766831


In [6]:
# DECISÃO DE LIMPEZA - filmes que não possuem reviews
# Como alguns filmes não possuem reviews, eles não se encaixam no modelo que buscamos, por isso, precisamos removê-los
filmes_com_rating = set(ratings_clean["movieId"])
filmes_sem_rating = movies_clean[~movies_clean["movieId"].isin(filmes_com_rating)]
movies_clean = movies_clean[movies_clean["movieId"].isin(filmes_com_rating)].copy()

print(f"Filmes removidos por ausência de avaliação: {len(movies) - len(movies_clean):,}")
print(f"Avaliações restantes: {len(movies_clean):,}")
print(f"Porcentagem descartada: {(len(movies) - len(movies_clean))/len(movies_clean)*100}")

Filmes removidos por ausência de avaliação: 56
Avaliações restantes: 9,686
Porcentagem descartada: 0.5781540367540781


In [7]:
# Salva os datasets limpos:

movies_clean.to_parquet('data/movies_clean.parquet', index=False)
ratings_clean.to_parquet('data/ratings_clean.parquet', index=False)